In [704]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, FloatType, DoubleType
spark = (
    SparkSession.builder
    .appName("Report")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyternb")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "512m")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")
irate_schema = StructType([
    StructField("Date", DateType()),
    StructField("Interest Rate", FloatType())
])

CPI_schema = StructType([
    StructField("Date", DateType()),
    StructField("CPI Value", FloatType())
])

irate = spark.read.options(delimiter=",", header=True).schema(irate_schema).csv('./interest_rate.csv')
CPI = spark.read.options(delimiter=",", header=True).schema(CPI_schema).csv('./US_inflation_rates.csv')

watch_schema = StructType([
    StructField("sale_id", StringType()),
    StructField("watch_id", StringType()),
    StructField("sale_date", DateType()),
    StructField("auction_year", IntegerType()),
    StructField("auction_month", IntegerType()),
    StructField("auction_house", StringType()),
    StructField("sale_location", StringType()),
    StructField("lot_number", StringType()),
    StructField("currency", StringType()),
    StructField("fx_rate_to_usd", DoubleType()),
    StructField("estimate_low_local", DoubleType()),
    StructField("estimate_high_local", DoubleType()),
    StructField("price_realized_local", DoubleType()),
    StructField("estimate_low_usd", DoubleType()),
    StructField("estimate_high_usd", DoubleType()),
    StructField("price_realized_usd", DoubleType()),
    StructField("estimate_beat_pct", DoubleType()),
    StructField("premium_over_retail_pct", DoubleType()),
    StructField("lot_condition", StringType()),
    StructField("box_and_papers", StringType()),
])

watch_auctions = spark.read.options(delimiter=",", header=True).schema(watch_schema).csv('./luxury_watch_auction_sales.csv')

crypto_schema = StructType([
    StructField("Open", FloatType()),
    StructField("High", FloatType()),
    StructField("Volume", DoubleType()),
    StructField("Close", FloatType()),
    StructField("Low", FloatType()),
    StructField("timestamp", IntegerType()),
    StructField("Ticker", StringType())
])

crypto = spark.read.options(delimiter=",", header=True).schema(crypto_schema).csv('./archive/*.csv')

Preprocessing

In [705]:
from pyspark.sql import Window, functions as sf
time_period = watch_auctions.agg(sf.min("sale_date").alias("min_date"), sf.max("sale_date").alias("max_date")).first()
min_date, max_date = time_period["min_date"], time_period["max_date"]

w = Window.orderBy("Date")
# Year over Year Inflation monthly
prev = sf.lag("CPI Value", 12).over(w)
CPI = CPI.withColumn("Inflation Percent Year/Year", sf.when(prev != 0, ((sf.col("CPI Value") - prev)/ prev) * 100))

rates = irate.join(CPI, on="Date", how="inner")
watch_auctions = watch_auctions.withColumn("sale_date", sf.to_date("sale_date"))


crypto = crypto.withColumn("date", sf.from_unixtime("timestamp"))
# crypto.take(4)

In [706]:
filtered_crypto = (
    crypto
    .where(crypto.date <= max_date)
    .where(crypto.date >= min_date)
)

# Crypto were any value is null during the period (full day with 0 trading volume)
# Crypto worth under 5 dollars at any given point also removed they are often scams/pump and dumps or irrelevant to the broader market
bad_ticker = (
    crypto.where((" OR ".join(f"{c} IS NULL" for c in filtered_crypto.columns)) + " OR Close < 5.0")
    .select("Ticker")
    .distinct()
)
filtered_crypto = filtered_crypto.join(bad_ticker, on="Ticker", how="left_anti")
Total_used = filtered_crypto.select("Ticker").distinct().count()

In [707]:
features = ["Close", "Volume", "date"]
mean_crypto = filtered_crypto.select(features).groupBy("date").agg({"Volume": "avg", "Close": "median"})
mean_crypto = mean_crypto.orderBy("date")

popular_crypto = ["BTC-USD", "ETH-USD"]
popular_crypto_df = (
    crypto.where(sf.col("Ticker").isin(popular_crypto))
    .select(features + ["Ticker"])
    .groupBy("date")
    .pivot("Ticker", popular_crypto)
    .agg(sf.first("Close").alias("Close"), sf.first("Volume").alias("Volume"))
)

crypto_dataset = mean_crypto.join(popular_crypto_df, on="date", how="inner")
crypto_dataset = crypto_dataset.orderBy("date")
crypto_dataset.plot.line(x="date", y=["median(Close)", "BTC-USD_Close", "ETH-USD_Close"], title="Price of Various Digital Currencies")


In [709]:
mean_monthly = (watch_auctions.select("auction_month", "auction_year", "estimate_beat_pct", "price_realized_usd")
                .groupBy(["auction_month", "auction_year"]).agg({"estimate_beat_pct": "avg", "*": "count", "price_realized_usd": "avg"}))
mean_monthly = mean_monthly.orderBy(["auction_year", "auction_month"])
mean_monthly = mean_monthly.withColumn("month", sf.make_date("auction_year", "auction_month", sf.lit(1)))
mean_monthly.plot.line(x="month", y=["avg(price_realized_usd)"], title="Avg Monthly Watch Price")

In [710]:
watch_auctions = watch_auctions.withColumn("log_price_realized_usd", sf.log(watch_auctions.price_realized_usd)).withColumn("log_estimate_usd", sf.log(watch_auctions.estimate_high_usd))
watch_auctions.plot.scatter(x="sale_date", y=["log_price_realized_usd", "log_estimate_usd"], title="Price Estimates of Watch vs Realized Price of the Watch")

In [711]:
df = watch_auctions.withColumnRenamed("sale_date", "date").join(crypto_dataset, on="date", how="inner")
df = df.withColumn("month", sf.trunc("Date", "month"))
rates = rates.withColumnRenamed("Date", "month")
df = df.join(rates, on="month", how="inner")

Convert lot_conditions into a discrete variable as it has a clear hierarchy but leave box_and_papers as categroical, also convert watch-id to a categrorical variables

In [712]:
print(watch_auctions.select("box_and_papers").distinct().collect())
print(watch_auctions.select("lot_condition").distinct().collect())
map_conditions = sf.create_map(
    sf.lit("Mint / Unworn"), sf.lit(3),
    sf.lit("Near Mint"), sf.lit(2),
    sf.lit("Very Good"), sf.lit(1),
    sf.lit("Good (Serviced)"), sf.lit(0),
)

df = df.withColumn("lot_condition", map_conditions[sf.col("lot_condition")].cast(IntegerType()))
splits = df.randomSplit(weights=[0.7, 0.3], seed=64)
train_df, test_df = splits[0], splits[1]

[Row(box_and_papers='Extract from Archives'), Row(box_and_papers='Complete (Box & Papers)'), Row(box_and_papers='Watch Only'), Row(box_and_papers='Papers Only')]
[Row(lot_condition='Very Good'), Row(lot_condition='Near Mint'), Row(lot_condition='Mint / Unworn'), Row(lot_condition='Good (Serviced)')]


We will now create 2 Random Forest Regression Models One without the Broader financial context and one with.

In [713]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.feature import VectorAssembler, StringIndexer, VectorIndexer
from pyspark.ml import Pipeline
categorical_features = ["box_and_papers", "sale_location"]
cont_features = ["log_estimate_usd", "lot_condition"]
pred_var = "log_price_realized_usd"

indxr = [
    StringIndexer(inputCol=col, outputCol=col + "_index", handleInvalid="keep")
    for col in categorical_features
]

b_assembler = VectorAssembler(
    inputCols=[f"{col}_index" for col in categorical_features] + cont_features,
    outputCol="features",
    handleInvalid="keep",
)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=pred_var,
)

rf_pipeline = Pipeline(stages=indxr + [b_assembler, rf])
rf_model = rf_pipeline.fit(train_df)

In [714]:
from pyspark.ml.evaluation import RegressionEvaluator
predictions = rf_model.transform(test_df)
evaluator = RegressionEvaluator(labelCol="log_price_realized_usd", metricName="rmse")
RMSE = evaluator.evaluate(predictions)
cont_features = ["log_estimate_usd", "lot_condition",  "median(Close)", "BTC-USD_Close", "Inflation Percent Year/Year", "Interest Rate"]

f_assembler = VectorAssembler(
    inputCols=[f"{col}_index" for col in categorical_features] + cont_features,
    outputCol="features",
    handleInvalid="keep",
)

fin_rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=pred_var,
)

fin_rf_pipeline = Pipeline(stages=indxr + [f_assembler, fin_rf])
fin_model = fin_rf_pipeline.fit(train_df)
fin_predictions = fin_model.transform(test_df)
fin_RMSE = evaluator.evaluate(fin_predictions)
print(f"MSE: {RMSE}\nFinancial RMSE: {fin_RMSE}")

MSE: 0.2812965490437814
Financial RMSE: 0.2906029948349338


In [715]:
predictions = predictions.withColumnRenamed("prediction", "baseline_prediction")
fin_predictions.join(predictions.select(["date", "baseline_prediction"]),on="date",how="inner").orderBy("date").plot.scatter(x="date", y=["log_price_realized_usd", "prediction", "baseline_prediction"], title="log(Predicted Prices) vs log(actual Prices)\nUsing Random Forest Regression")

This is not a statistically significant difference in RMSE and the small difference may be explained through leakage of date data through the finance variables. This could be resolved through the use of a time based split but since we only have data for the watches over 2016-2024 we do not have another significant low interest rate period to compare/train against.\
We will now introduce the use of Gradient Boosting to attempt to better amplify the difference in these sc

In [716]:
from pyspark.ml.regression import GBTRegressor
gbtr = GBTRegressor(
    featuresCol="features",
    labelCol=pred_var
)
gbtr_pipeline = Pipeline(stages=indxr + [b_assembler, gbtr])
base_gbtr = gbtr_pipeline.fit(train_df)
predictions = base_gbtr.transform(test_df)
bgbtr_RMSE = evaluator.evaluate(predictions)


fin_gbtr = GBTRegressor(
    featuresCol="features",
    labelCol=pred_var
)

fin_gbtr_pipeline = Pipeline(stages=indxr + [f_assembler, fin_gbtr])
fin_gbtr = fin_gbtr_pipeline.fit(train_df)
fin_predictions = fin_gbtr.transform(test_df)
fin_gbtr_RMSE = evaluator.evaluate(fin_predictions)
print(f"RMSE: {bgbtr_RMSE}\nFinancial RMSE: {fin_gbtr_RMSE}")

RMSE: 0.16030704867195386
Financial RMSE: 0.16507764852037513


In [717]:
predictions = predictions.withColumnRenamed("prediction", "baseline_prediction")
fin_predictions = fin_predictions.withColumnRenamed("prediction", "Financial prediction")
fin_predictions.join(predictions.select(["date", "baseline_prediction"]),on="date",how="inner").orderBy("date").plot.scatter(x="date", y=["log_price_realized_usd", "Financial prediction", "baseline_prediction"], title="Predicted prices vs Actual Price\nUsing Gradient Boosted Tree Regression")

This improves the predictions of the models but they still perform nearly identically so we cannot say that the addition of finacial conditions have meaningfully impacted the price of watches

In [718]:
base_gbtr.stages[-1].featureImportances
import pandas as pd
feat_vals = pd.DataFrame(columns=["model"] + categorical_features + cont_features)
models = [
    ("Base RF", rf_model),
    ("Financial RF", fin_model),
    ("Base GBTR", base_gbtr),
    ("Financial GBTR", fin_gbtr),
]
for i, (title, model) in enumerate(models):
    vals = model.stages[-1].featureImportances.toArray()
    if len(vals) < 8:
        vals = list(vals) + [0 for _ in range(8 - len(vals))]
    feat_vals.loc[i] = [title] + list(vals)
feat_vals.set_index("model", inplace=True)
feat_vals

,box_and_papers,sale_location,log_estimate_usd,lot_condition,median(Close),BTC-USD_Close,Inflation Percent Year/Year,Interest Rate
model,,,,,,,,
Base RF,0.006816,0.007094,0.979273,0.006817,0.000000,0.000000,0.000000,0.000000
Financial RF,0.004082,0.003193,0.939116,0.003033,0.008121,0.027468,0.007995,0.006993
Base GBTR,0.004893,0.006421,0.983932,0.004754,0.000000,0.000000,0.000000,0.000000
Financial GBTR,0.004712,0.004565,0.966123,0.006235,0.004915,0.004248,0.004284,0.004917


We can see from these models that the greatest predictor of the price is of course the estimated price, this is to be expected and is required to give these models an idea of the value of the watch not considering the economic conditions. We will now remove it from the table and consider the value the remaining effects have.

In [719]:
feat_vals.drop("log_estimate_usd", axis=1, inplace=True)
feat_vals = feat_vals.div(feat_vals.sum(axis=1), axis=0)

feat_vals

,box_and_papers,sale_location,lot_condition,median(Close),BTC-USD_Close,Inflation Percent Year/Year,Interest Rate
model,,,,,,,
Base RF,0.328836,0.342281,0.328883,0.000000,0.000000,0.000000,0.000000
Financial RF,0.067046,0.052447,0.049811,0.133377,0.451144,0.131315,0.114861
Base GBTR,0.304504,0.399626,0.295870,0.000000,0.000000,0.000000,0.000000
Financial GBTR,0.139105,0.134743,0.184059,0.145095,0.125391,0.126460,0.145147


We can see that in models where the Financial Statistics are available they make significant impacts on the model however we can also see from the MSE that the model performs with near identical performance. This could be explained by the aforementioned time leakage from the financial data so the model can timestamp when the watch was sold and overfit to drifts in watch pricing. Therefore we cannot show a meaningful relationship between the watch auction prices and the broader financial conditions.